In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
import os
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier


In [2]:
# only needs to be run once to give colab permission to access google drive
# This is to avoid needing to re-upload the 550 MB .csv every time

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#You might need to adjust the filepath for this, I haven't tested running it from another google account. I think since it's a shared folder is should still work?
path = "/content/drive/MyDrive/DATA-433 Group Project/Traffic_Crashes_-_Crashes.csv"

df = pd.read_csv(path)

In [4]:
df.columns = df.columns.str.strip()

print(df.shape)
print(df.columns[:10])

(1031725, 48)
Index(['CRASH_RECORD_ID', 'CRASH_DATE_EST_I', 'CRASH_DATE',
       'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION',
       'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE',
       'TRAFFICWAY_TYPE'],
      dtype='object')


In [5]:
df["target"] = df["MOST_SEVERE_INJURY"].apply(
    lambda x: 0 if x == "NO INDICATION OF INJURY" else 1
)

In [6]:
# --- DROP LEAKAGE + USELESS COLUMNS---
drop_cols = [
    "MOST_SEVERE_INJURY",
    "INJURIES_TOTAL",
    "INJURIES_FATAL",
    "INJURIES_INCAPACITATING",
    "INJURIES_NON_INCAPACITATING",
    "INJURIES_REPORTED_NOT_EVIDENT",
    "INJURIES_NO_INDICATION",
    "INJURIES_UNKNOWN",
    "CRASH_RECORD_ID",
    "LOCATION",
    "DATE_POLICE_NOTIFIED",
    "REPORT_TYPE",
    "CRASH_TYPE",
    "CRASH_DATE"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [7]:
# --- ENCODE ---
encoders = {}
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [8]:
# --- FILL NA ---
df = df.fillna(-1)

# --- SPLIT ---
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
joblib.dump(encoders, "encoders.pkl")

['encoders.pkl']

In [10]:
files.download("encoders.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# smaller, compressed model
small_model = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

small_model.fit(X_train, y_train)

small_preds = small_model.predict(X_test)

print("=== SMALL MODEL REPORT ===")
print(classification_report(y_test, small_preds))

=== SMALL MODEL REPORT ===
              precision    recall  f1-score   support

           0       0.94      0.77      0.84    176424
           1       0.33      0.69      0.45     29921

    accuracy                           0.75    206345
   macro avg       0.63      0.73      0.65    206345
weighted avg       0.85      0.75      0.78    206345



In [12]:
joblib.dump(small_model, "crash_model_small.pkl", compress=3)

size_mb = os.path.getsize("crash_model_small.pkl") / (1024 * 1024)
print(f"Compressed model size: {size_mb:.2f} MB")

Compressed model size: 1.00 MB


In [13]:
tiny_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=20,
    random_state=42
)

tiny_model.fit(X_train, y_train)

tiny_preds = tiny_model.predict(X_test)

print("=== TINY MODEL REPORT ===")
print(classification_report(y_test, tiny_preds))

joblib.dump(tiny_model, "crash_model_tiny.pkl", compress=3)

size_mb = os.path.getsize("crash_model_tiny.pkl") / (1024 * 1024)
print(f"Tiny model size: {size_mb:.2f} MB")

=== TINY MODEL REPORT ===
              precision    recall  f1-score   support

           0       0.89      0.99      0.93    176424
           1       0.79      0.25      0.38     29921

    accuracy                           0.88    206345
   macro avg       0.84      0.62      0.66    206345
weighted avg       0.87      0.88      0.85    206345

Tiny model size: 0.02 MB


In [14]:
files.download("crash_model_small.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
balanced_model = RandomForestClassifier(
    n_estimators=30,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [16]:
balanced_model.fit(X_train, y_train)

balanced_preds = balanced_model.predict(X_test)

print("=== BALANCED MODEL REPORT ===")
print(classification_report(y_test, balanced_preds))

=== BALANCED MODEL REPORT ===
              precision    recall  f1-score   support

           0       0.94      0.79      0.85    176424
           1       0.35      0.69      0.47     29921

    accuracy                           0.77    206345
   macro avg       0.64      0.74      0.66    206345
weighted avg       0.85      0.77      0.80    206345



In [17]:
joblib.dump(balanced_model, "crash_model_balanced.pkl", compress=3)

size_mb = os.path.getsize("crash_model_balanced.pkl") / (1024 * 1024)
print(f"Compressed model size: {size_mb:.2f} MB")

Compressed model size: 3.65 MB


In [18]:
files.download("crash_model_balanced.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>